# Chapter 26 — Transformer II: Position Encoding, Complexity & Pre-Training (BERT, GPT)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part4_transformers_genai/ch26_transformer_ii/ch26_notebook.ipynb)

Companion notebook for **Chapter 26** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Attention has no built-in notion of order, so position information must be added explicitly to the embeddings.
- Sinusoidal position encodings stay bounded and generalise to sequences longer than any seen during training.
- Parameter count grows with the square of the model dimension d and linearly with the number of layers N.
- Pre-training on huge unlabeled corpora followed by fine-tuning on a small labelled set is the dominant modern recipe.
- BERT pre-trains the encoder with masked language modelling; GPT pre-trains the decoder to predict the next word.

## Example 1: Sinusoidal Position Encoding

This builds the position encoding matrix using the exact formula, with sin for even indices and cos for odd indices. The output shows that every position gets a distinct vector, that the same word embedding becomes different at different positions, and that the dot product between position vectors decays with distance - which is how the model infers relative order.

In [1]:
# Chapter 26 - Example 1: Sinusoidal Position Encoding
import numpy as np
np.set_printoptions(suppress=True)

def position_encoding(T, d):
    """p_t[i] = sin(w_k * t) if i=2k ; cos(w_k * t) if i=2k+1, w_k = 1/10000^(2k/d)"""
    P = np.zeros((T, d))
    for t in range(T):
        for i in range(d):
            k  = i // 2
            wk = 1.0 / (10000 ** (2 * k / d))
            P[t, i] = np.sin(wk * t) if i % 2 == 0 else np.cos(wk * t)
    return P

print("=== Sinusoidal Position Encoding ===\n")
print("Why needed? Attention has NO notion of order. Consider:")
print('  "The food was good, not bad at all"')
print('  "The food was bad, not good at all"')
print("Same words, opposite meaning -> position information is essential.\n")

T, d = 6, 8
P = position_encoding(T, d)
print(f"Position encoding matrix p_t for T={T} positions, d={d}:")
print(np.round(P, 4))

print("\nEvery position gets a DIFFERENT vector:")
for t in range(3):
    print(f"  p_{t} = {np.round(P[t], 4)}")

print("\nCombine with word embedding:  x_hat_t = x_t + p_t")
np.random.seed(0)
x = np.round(np.random.randn(d), 3)
print(f"  word embedding x  (same word at any position): {x}")
print(f"  x_hat at position 0 = {np.round(x + P[0], 3)}")
print(f"  x_hat at position 3 = {np.round(x + P[3], 3)}")
print("  -> the SAME word gets different embeddings at different positions.")

print("\n=== Property: similarity decays with distance ===")
print("Dot product between position 0 and position t:")
for t in range(T):
    print(f"  p_0 . p_{t} = {P[0] @ P[t]:+.4f}")
print("Nearby positions have higher similarity -> the model can infer relative order.")

print("\n=== Why sinusoidal (not learned integers)? ===")
print("  * Values stay bounded in [-1, 1] regardless of sequence length")
print("  * Generalises to sequences longer than those seen in training")
print("  * Relative position is expressible as a linear function of p_t")

=== Sinusoidal Position Encoding ===

Why needed? Attention has NO notion of order. Consider:
  "The food was good, not bad at all"
  "The food was bad, not good at all"
Same words, opposite meaning -> position information is essential.

Position encoding matrix p_t for T=6 positions, d=8:
[[ 0.      1.      0.      1.      0.      1.      0.      1.    ]
 [ 0.8415  0.5403  0.0998  0.995   0.01    1.      0.001   1.    ]
 [ 0.9093 -0.4161  0.1987  0.9801  0.02    0.9998  0.002   1.    ]
 [ 0.1411 -0.99    0.2955  0.9553  0.03    0.9996  0.003   1.    ]
 [-0.7568 -0.6536  0.3894  0.9211  0.04    0.9992  0.004   1.    ]
 [-0.9589  0.2837  0.4794  0.8776  0.05    0.9988  0.005   1.    ]]

Every position gets a DIFFERENT vector:
  p_0 = [0. 1. 0. 1. 0. 1. 0. 1.]
  p_1 = [0.8415 0.5403 0.0998 0.995  0.01   1.     0.001  1.    ]
  p_2 = [ 0.9093 -0.4161  0.1987  0.9801  0.02    0.9998  0.002   1.    ]

Combine with word embedding:  x_hat_t = x_t + p_t
  word embedding x  (same word at any po

## Example 2: Transformer Parameter-Count Calculator

This turns the complexity formulas into working code: 3ldhN for each MHA group, 3d^2N for the W_o matrices, 4d^2N for the PWFFNN sub-layers and dV for the decoder's linear layer. Applying it to the original base configuration (d=512, h=8, N=6) gives about 44 million parameters, and the comparison table shows that doubling d roughly quadruples the count while doubling N only doubles it.

In [2]:
# Chapter 26 - Example 2: Transformer parameter-count calculator
print("=== Complexity of the Transformer Model ===\n")
print("Parameter counts:")
print("  Encoder self-attention MHA in N layers : 3*l*d*h*N")
print("  Decoder self-attention MHA in N layers : 3*l*d*h*N")
print("  Cross-attention MHA in N layers        : 3*l*d*h*N")
print("  W_o matrices in 3 MHA sub-layers       : 3*d^2*N")
print("  Two weight matrices in 2 PWFFNN        : 4*d^2*N")
print("  Linear layer in decoder                : d*V\n")

def transformer_params(d, h, N, V, verbose=True):
    l = d // h                      # per-head dimension
    enc_mha   = 3 * l * d * h * N
    dec_mha   = 3 * l * d * h * N
    cross_mha = 3 * l * d * h * N
    wo        = 3 * d * d * N
    pwffnn    = 4 * d * d * N
    linear    = d * V
    total = enc_mha + dec_mha + cross_mha + wo + pwffnn + linear
    if verbose:
        print(f"Configuration: d={d}, h={h} heads (l={l}), N={N} layers, vocab V={V}")
        print(f"  Encoder self-attention MHA : {enc_mha:>12,}")
        print(f"  Decoder self-attention MHA : {dec_mha:>12,}")
        print(f"  Cross-attention MHA        : {cross_mha:>12,}")
        print(f"  W_o matrices               : {wo:>12,}")
        print(f"  PWFFNN weight matrices     : {pwffnn:>12,}")
        print(f"  Decoder linear layer       : {linear:>12,}")
        print(f"  {'TOTAL':<27}{total:>12,}\n")
    return total

# The original "Attention is all you need" base configuration
transformer_params(d=512, h=8, N=6, V=37000)

print("=== How parameters scale with each hyperparameter ===\n")
print(f"{'Config':<38} {'Parameters':>14}")
print("-" * 54)
for (d, h, N, V, label) in [
    (256, 4, 2, 10000, "small  (d=256, h=4,  N=2)"),
    (512, 8, 6, 37000, "base   (d=512, h=8,  N=6)"),
    (1024,16, 6, 37000, "large  (d=1024,h=16, N=6)"),
    (512, 8,12, 37000, "deep   (d=512, h=8,  N=12)"),
]:
    p = transformer_params(d, h, N, V, verbose=False)
    print(f"  {label:<36} {p:>14,}")

print("\nObservations:")
print("  * Doubling d roughly QUADRUPLES the parameters (terms are O(d^2)).")
print("  * Doubling N doubles the parameters (linear in depth).")
print("  * Changing h alone does not change the count, since l = d/h keeps l*h = d.")

print("\n=== Training the Transformer ===")
print("  * Trained end-to-end with error backpropagation, cross-entropy loss.")
print("  * Training data for translation: (source sentence, target sentence) pairs.")
print("  * Desired output at time t: one-hot vector of the t-th target word.")
print("  * Error is backpropagated through decoder then encoder layers, N, N-1, ..., 1.")

=== Complexity of the Transformer Model ===

Parameter counts:
  Encoder self-attention MHA in N layers : 3*l*d*h*N
  Decoder self-attention MHA in N layers : 3*l*d*h*N
  Cross-attention MHA in N layers        : 3*l*d*h*N
  W_o matrices in 3 MHA sub-layers       : 3*d^2*N
  Two weight matrices in 2 PWFFNN        : 4*d^2*N
  Linear layer in decoder                : d*V

Configuration: d=512, h=8 heads (l=64), N=6 layers, vocab V=37000
  Encoder self-attention MHA :    4,718,592
  Decoder self-attention MHA :    4,718,592
  Cross-attention MHA        :    4,718,592
  W_o matrices               :    4,718,592
  PWFFNN weight matrices     :    6,291,456
  Decoder linear layer       :   18,944,000
  TOTAL                        44,109,824

=== How parameters scale with each hyperparameter ===

Config                                     Parameters
------------------------------------------------------
  small  (d=256, h=4,  N=2)                 4,657,152
  base   (d=512, h=8,  N=6)          

## Example 3: BERT Masked LM vs GPT Autoregressive Pre-Training

This contrasts the two pre-training objectives directly. BERT masks about 15% of tokens and predicts them using both left and right context; GPT predicts each next word from the preceding words only. The final section computes the cross-entropy loss for a single prediction, showing how a confident wrong answer is penalised far more heavily than an uncertain one.

In [3]:
# Chapter 26 - Example 3: BERT Masked LM vs GPT Autoregressive pre-training
import numpy as np
np.random.seed(3)

sentence = "the quick brown fox jumps over the lazy dog".split()
T = len(sentence)

print("=== Pre-Training Objectives: BERT vs GPT ===\n")
print("Both pre-train on huge UNLABELED text, then fine-tune on a small")
print("labeled dataset for a downstream task.\n")
print(f"Sentence ({T} tokens): {' '.join(sentence)}\n")

# ---------- BERT: Masked Language Modelling ----------
print("--- BERT (encoder pre-training): Masked Language Modelling ---")
print("Randomly mask ~15% of tokens; the model predicts them using BOTH")
print("left and right context (bidirectional).\n")
n_mask = max(1, int(0.15 * T))
mask_idx = sorted(int(i) for i in np.random.choice(T, n_mask, replace=False))
masked_sentence = [("[MASK]" if i in mask_idx else w) for i, w in enumerate(sentence)]
print(f"  Masked positions : {mask_idx}")
print(f"  Model input      : {' '.join(masked_sentence)}")
print(f"  Prediction target: {[sentence[i] for i in mask_idx]}")
print("  Context used     : ALL other tokens (both directions)\n")

print("  Second BERT objective - Next Sentence Prediction (NSP):")
print("    Input : [CLS] the quick brown fox ... [SEP] it was very fast [SEP]")
print("    Target: IsNext = True/False\n")

# ---------- GPT: Autoregressive LM ----------
print("--- GPT (decoder pre-training): Autoregressive Language Modelling ---")
print("Predict the NEXT word using only the PRECEDING words (unidirectional).\n")
print(f"  {'t':>2}  {'Input (context)':<45} {'Predict'}")
print("  " + "-" * 66)
for t in range(1, min(T, 6)):
    ctx = " ".join(sentence[:t])
    print(f"  {t:>2}  {ctx:<45} -> {sentence[t]}")
print("  ...")

# ---------- Cross-entropy loss for one prediction ----------
print("\n=== Cross-Entropy Loss for One Prediction ===\n")
vocab = ["fox", "dog", "cat", "car", "tree"]
logits = np.array([3.2, 1.1, 0.8, -0.5, 0.2])
e = np.exp(logits - logits.max()); probs = e / e.sum()
target = 0   # correct word is "fox"

print(f"Vocabulary      : {vocab}")
print(f"Model logits    : {logits}")
print(f"Softmax probs   : {np.round(probs, 4)}")
print(f"True next word  : '{vocab[target]}' (one-hot target)")
loss = -np.log(probs[target])
print(f"Cross-entropy loss = -log(p_correct) = -log({probs[target]:.4f}) = {loss:.4f}")

print("\nIf the model were confident and WRONG (p_correct = 0.01):")
print(f"  loss = -log(0.01) = {-np.log(0.01):.4f}   (heavily penalised)")
print("If the model were perfect (p_correct = 1.0):")
print(f"  loss = -log(1.00) = {-np.log(1.0):.4f}")

print("\n=== Summary ===")
print("  BERT : encoder-only,  bidirectional, Masked LM + NSP")
print("  GPT  : decoder-only,  autoregressive (masked self-attention)")
print("  T5   : encoder-decoder, text-to-text transfer")
print("  ViLBERT: encoder pre-training for visual-linguistic (image+text) data")

=== Pre-Training Objectives: BERT vs GPT ===

Both pre-train on huge UNLABELED text, then fine-tune on a small
labeled dataset for a downstream task.

Sentence (9 tokens): the quick brown fox jumps over the lazy dog

--- BERT (encoder pre-training): Masked Language Modelling ---
Randomly mask ~15% of tokens; the model predicts them using BOTH
left and right context (bidirectional).

  Masked positions : [6]
  Model input      : the quick brown fox jumps over [MASK] lazy dog
  Prediction target: ['the']
  Context used     : ALL other tokens (both directions)

  Second BERT objective - Next Sentence Prediction (NSP):
    Input : [CLS] the quick brown fox ... [SEP] it was very fast [SEP]
    Target: IsNext = True/False

--- GPT (decoder pre-training): Autoregressive Language Modelling ---
Predict the NEXT word using only the PRECEDING words (unidirectional).

   t  Input (context)                               Predict
  ------------------------------------------------------------------
  